[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/00_api_access.ipynb)

# Part 0 — Getting API access

In Agentic Programming, we write programs that are able to send text to a large language model and read/interact with its reply. To do that from Python you need an **API key** — a password-like string that identifies your account to the model provider.

This is a different way of using the model than running Gemini/Claude/ChatGPT in a chat windows like you may be accustomed to. Those are built for a person typing in a browser while an API is built for a program. If we are going to build autonomous workflows, everything should be handled within a program.

**What does this notebook do?** It shows how to set up your API key using Google's AI studio. We use this for the purposes of this crash course because they offer a zero-cost API key which is ideal for students and people getting started. This notebook: installs the Google Gemini package, loads your key, builds a `client` object the other notebooks reuse, and sends one short prompt to check it works.

**You are done when** the last cell prints a one-sentence response to a prompt asking the LLM to say hello.

## 0.0 Getting a key

1. Go to <https://aistudio.google.com/apikey> and sign in with any Google account.
2. Click **Create API key** and copy the string it gives you.
3. In Colab, open the **Secrets** panel (key icon, left sidebar), choose *Add new secret*, name it `GEMINI_API_KEY`, paste the key as the value, and turn on *Notebook access*.

Running locally instead of Colab? Set it as an environment variable: `export GEMINI_API_KEY=...`

**Do not paste your key into a code cell.** Whatever you type into a cell is saved inside the `.ipynb` file, so the key would travel with the notebook into git or anywhere you share it. The Secrets panel keeps it outside the file.

## 0.1 Install the Gemini package
Colab already has most of what we need, so this only adds what is missing. Note that you have to be a little careful to match the version of `google-genai` to the `google-auth` that is used by default in Colab.

In [ ]:
# Install dependencies (run once).
import sys
if 'google.colab' in sys.modules:
    %pip install -U -q "google-genai<2.13" "google-auth==2.49.0"


## 0.2 Read your key and select the LLM model

If this raises an error, the key is not set correctly — go back to the Secrets panel and make sure it is set consistently.

In [ ]:
import os, time
from google import genai
from google.genai import types as gtypes

# Load the API key. In Colab use the Secrets panel; locally use an environment variable.
try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

client = genai.Client(api_key=API_KEY)
# Free-tier limits for this model, confirmed Sept 2026: 15 RPM, 250k TPM, 1000 RPD.
# Agent loops are launch rapid-fire prompts, so the per-minute cap is what the 
# retry wrapper below handles. If you violate the per-day cap, the model will
# return a 429 error and the session will end, forcing you to set up a new key.

MODEL = "gemini-3.1-flash-lite"

## 0.3 The retry wrapper

The free tier allows a limited number of requests per minute. If we exceed it the API returns an error instead of an answer, so this wrapper waits and tries again rather than failing.

In [ ]:
def generate_with_retry(*, contents, config=None, max_attempts=6):
    """client.models.generate_content with exponential backoff on 429.

    Free-tier Gemini caps requests/minute. A ReAct loop can fire many calls
    back-to-back and trip the limit; we sleep and retry instead of crashing.
    """
    delay = 4.0
    for attempt in range(max_attempts):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            if "429" not in msg and "RESOURCE_EXHAUSTED" not in msg and "quota" not in msg.lower():
                raise
            if attempt == max_attempts - 1:
                raise
            print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s and retrying...")
            time.sleep(delay)
            delay = min(delay * 2, 60.0)

print(f"Gemini client ready (model={MODEL}).")

## 0.4 Final test

The following is how you ask the model a question. This is identical to what you may be familiar with talking with an LLM through a chat window. If everything is set up correctly, the model should say hello to you.

In [ ]:
# Smoke test: one chat call, no tools.
resp = generate_with_retry(
    contents="Say hello in one short sentence.",
)
print(resp.text)

## 0.5 Available models

We use `gemini-3.1-flash-lite` because it has the most generous free-tier limits. We need to be careful because although it is free, there is a response per minute ceiling that we can easily hit. If at any point you get the error

```
ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message':
  'You exceeded your current quota ...', 'status': 'RESOURCE_EXHAUSTED'}}
```

it means that you have exceeded the limits. At the time that we prepared this, `gemini-3.1-flash-lite` allowed 15 requests per minute, 250,000 tokens per minute, and 1,000 requests per day. **Expect these to change.** Google sets them per account rather than publishing a fixed table, so check the limits actually applied to your key at <https://aistudio.google.com/rate-limit>.

The cell below will print what models are available under your key if you'd like to try other models.

In [ ]:
for m in client.models.list():
    acts = m.supported_actions or []
    if acts and "generateContent" not in acts:
        continue
    print(f"{m.name:42} in={m.input_token_limit or '?':>9} out={m.output_token_limit or '?':>7}")

## 0.6 Using OpenAI or Anthropic instead

> **Untested.** Everything in these notebooks was written and run against Gemini. The
> snippets below are correct as far as each provider's documented API goes, but we have
> not run the workshop material through them. If something breaks with these,
> please let us know so that we can fix this material.

To swap out the Gemini backend, we only need to bring in the alternative `anthropic`/`openai` libraries and redefine the function that pushes a text prompt into the model. Below are blocks of code that are equivalent to what we ran above.

### Anthropic

In [ ]:
# --- Anthropic ---------------------------------------------------------------
# Needs ANTHROPIC_API_KEY in the Colab Secrets panel (or your environment).
# Redefines client, MODEL and generate_with_retry, so everything above keeps working.
import os, sys, time
if 'google.colab' in sys.modules:
    %pip install -U -q anthropic

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')

if not ANTHROPIC_API_KEY:
    print("No ANTHROPIC_API_KEY found - add one to Colab Secrets to run this cell.")
else:
    import anthropic

    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    MODEL = "claude-haiku-4-5"          # also claude-sonnet-5, claude-opus-5

    class _Reply:
        """Stands in for Gemini's response object; the cells above only read .text"""
        def __init__(self, text):
            self.text = text

    def generate_with_retry(*, contents, config=None, max_attempts=6):
        """Same name, signature and .text as the Gemini helper above."""
        kwargs = {"model": MODEL,
                  "max_tokens": 1024,                   # required by this API
                  "messages": [{"role": "user", "content": contents}]}
        system = getattr(config, "system_instruction", None)
        if system:
            kwargs["system"] = system
        # No temperature: current Claude models reject sampling parameters.

        delay = 4.0
        for attempt in range(max_attempts):
            try:
                resp = client.messages.create(**kwargs)
                return _Reply("".join(b.text for b in resp.content if b.type == "text"))
            except (anthropic.RateLimitError, anthropic.InternalServerError):
                if attempt == max_attempts - 1:
                    raise
                print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s and retrying...")
                time.sleep(delay)
                delay = min(delay * 2, 60.0)

    print(generate_with_retry(contents="Say hello in one short sentence.").text)

### OpenAI

In [ ]:
# --- OpenAI ------------------------------------------------------------------
# Needs OPENAI_API_KEY in the Colab Secrets panel (or your environment).
# Redefines client, MODEL and generate_with_retry, so everything above keeps working.
import os, sys, time
if 'google.colab' in sys.modules:
    %pip install -U -q openai

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

if not OPENAI_API_KEY:
    print("No OPENAI_API_KEY found - add one to Colab Secrets to run this cell.")
else:
    import openai

    client = openai.OpenAI(api_key=OPENAI_API_KEY)
    MODEL = "gpt-5.6-luna"              # also gpt-5.6-terra, gpt-6-astra

    class _Reply:
        """Stands in for Gemini's response object; the cells above only read .text"""
        def __init__(self, text):
            self.text = text

    def generate_with_retry(*, contents, config=None, max_attempts=6):
        """Same name, signature and .text as the Gemini helper above."""
        messages = []
        system = getattr(config, "system_instruction", None)
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": contents})

        kwargs = {"model": MODEL, "messages": messages}
        temperature = getattr(config, "temperature", None)
        if temperature is not None:
            kwargs["temperature"] = temperature

        delay = 4.0
        for attempt in range(max_attempts):
            try:
                resp = client.chat.completions.create(**kwargs)
                return _Reply(resp.choices[0].message.content or "")
            except (openai.RateLimitError, openai.InternalServerError):
                if attempt == max_attempts - 1:
                    raise
                print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s and retrying...")
                time.sleep(delay)
                delay = min(delay * 2, 60.0)

    print(generate_with_retry(contents="Say hello in one short sentence.").text)

## 0.7 Further reading

For further details please refer to the appropriate APIs:

- **Gemini** — [API docs](https://ai.google.dev/gemini-api/docs) · [text generation](https://ai.google.dev/gemini-api/docs/text-generation) · [function calling](https://ai.google.dev/gemini-api/docs/function-calling)
- **Anthropic** — [Messages API](https://platform.claude.com/docs/en/api/messages) · [tool use](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview)
- **OpenAI** — [API docs](https://developers.openai.com/api/docs) · [quickstart](https://developers.openai.com/api/docs/quickstart) · [models](https://developers.openai.com/api/docs/models)